In [ ]:
import os
import numpy as np
import pandas as pd
import folium
import branca
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import glob
import math
import warnings
from datetime import datetime
import gc  # For garbage collection

warnings.filterwarnings('ignore')

# Use non-interactive backend for matplotlib
import matplotlib
matplotlib.use('Agg')

###########################################
# Configuration Parameters
###########################################

RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv',
    'DATA_DIR': 'data/citibike',
    'OUTPUT_DIR': 'results',
    'MODE': 'all',  # 'train', 'visualize', 'all'
    'MAX_STATIONS': 20,
    'SAMPLE_RATE': 1.0,
    'DISTANCE_THRESHOLD': 2.0,
    'ANALYZE_SEASONAL_PATTERNS': True,
    'CHUNK_SIZE': 10000,
}

# Time period configurations
TIME_PERIODS = {
    "Morning Peak (7-10 AM)": (7, 10),
    "Afternoon (10 AM-4 PM)": (10, 16),
    "Evening Peak (4-7 PM)": (16, 19),
    "All Day": (0, 24)
}

# Jersey City Zone Classification
ZONE_CLASSIFICATION_RULES = {
    'residential_keywords': [
        'heights', 'bergen', 'lafayette', 'greenville', 'mcginley', 
        'marion', 'duncan', 'communipaw', 'westside', 'journal square',
        # Add more residential indicators
        'park', 'school', 'housing', 'residential', 'neighborhood',
        'home', 'family', 'community', 'garden', 'terrace'
    ],
    'commercial_keywords': [
        'downtown', 'exchange', 'newport', 'harborside', 'grove', 
        'christopher columbus', 'essex', 'liberty', 'morris canal',
        'hudson', 'plaza', 'waterfront', 'financial',
        # Add more commercial indicators
        'office', 'business', 'center', 'station', 'terminal', 'transit',
        'mall', 'shopping', 'market', 'commercial', 'corporate',
        'bank', 'hotel', 'restaurant', 'store', 'shop'
    ]
}

# Neighborhood labels for enhanced visualization
NEIGHBORHOOD_LABELS = {
    'The Heights': (40.7489, -74.0464),
    'Downtown JC': (40.7178, -74.0431),
    'Exchange Place': (40.7161, -74.0336),
    'Journal Square': (40.7324, -74.0635),
    'Bergen-Lafayette': (40.7089, -74.0665),
    'Greenville': (40.7067, -74.0774)
}

###########################################
# Draggable Legend Utilities
###########################################

def create_draggable_html_legend(legend_items, title="Legend", position="top-right"):
    """Create a draggable HTML legend for Folium maps"""
    
    # Position mapping
    position_styles = {
        "top-right": "top: 10px; right: 10px;",
        "top-left": "top: 10px; left: 10px;",
        "bottom-right": "bottom: 10px; right: 10px;",
        "bottom-left": "bottom: 10px; left: 10px;",
    }
    
    legend_html = f'''
    <div id="draggable-legend" style="position: absolute; 
                                     {position_styles.get(position, position_styles["top-right"])}
                                     cursor: move;
                                     width: auto; 
                                     background-color: rgba(255, 255, 255, 0.95); 
                                     border: 2px solid #333; 
                                     border-radius: 8px;
                                     z-index: 9999; 
                                     font-size: 14px; 
                                     padding: 15px;
                                     box-shadow: 0 4px 12px rgba(0,0,0,0.3);
                                     font-family: Arial, sans-serif;
                                     user-select: none;">
        <div style="font-weight: bold; 
                    margin-bottom: 10px; 
                    text-align: center; 
                    font-size: 16px;
                    color: #333;
                    border-bottom: 1px solid #ccc;
                    padding-bottom: 8px;
                    cursor: move;">{title}</div>
        <div id="legend-content">
    '''
    
    for item in legend_items:
        if item['type'] == 'circle':
            legend_html += f'''
            <div style="margin: 8px 0; display: flex; align-items: center;">
                <div style="width: 16px; 
                           height: 16px; 
                           background-color: {item['color']}; 
                           border-radius: 50%; 
                           border: 2px solid black;
                           margin-right: 10px;
                           flex-shrink: 0;"></div>
                <span style="color: #333; font-weight: 500;">{item['label']}</span>
            </div>
            '''
        elif item['type'] == 'line':
            legend_html += f'''
            <div style="margin: 8px 0; display: flex; align-items: center;">
                <div style="width: 20px; 
                           height: {item.get('width', 3)}px; 
                           background-color: {item['color']}; 
                           margin-right: 10px;
                           flex-shrink: 0;"></div>
                <span style="color: #333; font-weight: 500;">{item['label']}</span>
            </div>
            '''
        elif item['type'] == 'square':
            legend_html += f'''
            <div style="margin: 8px 0; display: flex; align-items: center;">
                <div style="width: 16px; 
                           height: 16px; 
                           background-color: {item['color']}; 
                           border: 2px solid black;
                           margin-right: 10px;
                           flex-shrink: 0;"></div>
                <span style="color: #333; font-weight: 500;">{item['label']}</span>
            </div>
            '''
    
    legend_html += '''
        </div>
    </div>
    
    <script>
    // Make legend draggable
    (function() {
        var legend = document.getElementById('draggable-legend');
        var isDragging = false;
        var currentX;
        var currentY;
        var initialX;
        var initialY;
        var xOffset = 0;
        var yOffset = 0;

        function dragStart(e) {
            if (e.type === "touchstart") {
                initialX = e.touches[0].clientX - xOffset;
                initialY = e.touches[0].clientY - yOffset;
            } else {
                initialX = e.clientX - xOffset;
                initialY = e.clientY - yOffset;
            }

            if (e.target === legend || e.target.parentNode === legend || e.target.parentNode.parentNode === legend) {
                isDragging = true;
            }
        }

        function dragEnd(e) {
            initialX = currentX;
            initialY = currentY;
            isDragging = false;
        }

        function drag(e) {
            if (isDragging) {
                e.preventDefault();
                
                if (e.type === "touchmove") {
                    currentX = e.touches[0].clientX - initialX;
                    currentY = e.touches[0].clientY - initialY;
                } else {
                    currentX = e.clientX - initialX;
                    currentY = e.clientY - initialY;
                }

                xOffset = currentX;
                yOffset = currentY;

                setTranslate(currentX, currentY, legend);
            }
        }

        function setTranslate(xPos, yPos, el) {
            el.style.transform = "translate3d(" + xPos + "px, " + yPos + "px, 0)";
        }

        if (legend) {
            legend.addEventListener("mousedown", dragStart, false);
            legend.addEventListener("touchstart", dragStart, false);
            
            document.addEventListener("mouseup", dragEnd, false);
            document.addEventListener("touchend", dragEnd, false);
            
            document.addEventListener("mousemove", drag, false);
            document.addEventListener("touchmove", drag, false);
        }
    })();
    </script>
    '''
    
    return legend_html

def make_matplotlib_legend_draggable(ax, *args, **kwargs):
    """Create a draggable legend for matplotlib plots with enhanced styling"""
    # Extract custom styling parameters and add them to kwargs for legend creation
    frameon = kwargs.get('frameon', True)
    fancybox = kwargs.get('fancybox', True) 
    shadow = kwargs.get('shadow', True)
    framealpha = kwargs.get('framealpha', 0.9)
    facecolor = kwargs.get('facecolor', 'white')
    edgecolor = kwargs.get('edgecolor', 'black')
    
    # Set the styling parameters directly in kwargs
    kwargs['frameon'] = frameon
    kwargs['fancybox'] = fancybox
    kwargs['shadow'] = shadow
    kwargs['framealpha'] = framealpha
    kwargs['facecolor'] = facecolor
    kwargs['edgecolor'] = edgecolor
    
    # Create the legend with styling applied
    legend = ax.legend(*args, **kwargs)
    
    # Apply additional styling to the frame if needed
    if frameon and legend.get_frame():
        legend.get_frame().set_linewidth(1.5)
    
    # Make it draggable
    legend.set_draggable(True)
    return legend

###########################################
# Helper Functions
###########################################

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371
    
    return c * r

def classify_stations_by_zone_type(stations_df):
    """Classify stations into ONLY residential or commercial (no mixed category)"""
    zone_classification = {}
    
    for _, station in stations_df.iterrows():
        station_name_lower = station['name'].lower()
        
        is_commercial = any(keyword in station_name_lower 
                          for keyword in ZONE_CLASSIFICATION_RULES['commercial_keywords'])
        is_residential = any(keyword in station_name_lower 
                           for keyword in ZONE_CLASSIFICATION_RULES['residential_keywords'])
        
        # BINARY CLASSIFICATION: Force decision between residential and commercial
        if is_commercial and is_residential:
            # If both keywords found, prioritize commercial (business areas typically more specific)
            zone_classification[station['name']] = 'commercial'
        elif is_commercial:
            zone_classification[station['name']] = 'commercial'
        elif is_residential:
            zone_classification[station['name']] = 'residential'
        else:
            # For stations with no clear keywords, use geographic/contextual heuristics
            # You can customize this logic based on your knowledge of JC
            
            # Example heuristics (customize based on your local knowledge):
            # Downtown/waterfront areas tend to be commercial
            if any(word in station_name_lower for word in ['street', 'ave', 'avenue', 'path', 'plaza']):
                # Major streets and avenues are often commercial corridors
                zone_classification[station['name']] = 'commercial'
            elif any(word in station_name_lower for word in ['park', 'school', 'residential', 'housing']):
                # Areas near parks, schools tend to be residential
                zone_classification[station['name']] = 'residential'
            else:
                # Default fallback - you might want to customize this
                # Option 1: Default to residential (most bike stations serve residents)
                zone_classification[station['name']] = 'residential'
                
                # Option 2: Default to commercial (business/transit hubs)
                # zone_classification[station['name']] = 'commercial'
    
    return zone_classification

def get_flow_type(origin_station, dest_station, zone_classification):
    """Helper function to determine flow type"""
    if not zone_classification:
        return "Unknown"
    
    origin_type = zone_classification.get(origin_station, 'unknown')
    dest_type = zone_classification.get(dest_station, 'unknown')
    
    if origin_type == 'residential' and dest_type == 'commercial':
        return "Residential → Commercial"
    elif origin_type == 'commercial' and dest_type == 'residential':
        return "Commercial → Residential"
    elif origin_type == 'residential' and dest_type == 'residential':
        return "Residential → Residential"
    elif origin_type == 'commercial' and dest_type == 'commercial':
        return "Commercial → Commercial"
    else:
        return f"{origin_type.title()} → {dest_type.title()}"

###########################################
# Data Loading Functions
###########################################

def load_citibike_data(data_dir, data_files_pattern, sample_rate=1.0, chunk_size=10000):
    """Load and preprocess Citibike data from multiple files with memory management and robust timestamp parsing"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    file_pattern = os.path.join(data_dir, data_files_pattern)
    file_list = glob.glob(file_pattern)
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {file_pattern}")
    
    file_list.sort()
    print(f"Found {len(file_list)} files to process")
    
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        chunks = []
        chunk_count = 0
        
        try:
            for chunk in pd.read_csv(file_path, chunksize=chunk_size):
                if sample_rate < 1.0:
                    sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
                else:
                    sampled_chunk = chunk
                
                chunks.append(sampled_chunk)
                chunk_count += 1
                
                if chunk_count % 10 == 0:
                    print(f"  Processed {chunk_count} chunks...")
        
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            continue
        
        if chunks:
            monthly_trips = pd.concat(chunks, ignore_index=True)
            print(f"Loaded {len(monthly_trips)} trips from {file_name}")
            
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
            
            del chunks
            gc.collect()
        else:
            print(f"No data loaded from {file_name}")
    
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    print("Combining all monthly data...")
    trips_df = pd.concat(all_trips, ignore_index=True)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    del all_trips
    gc.collect()
    
    print("Cleaning data...")
    required_cols = ['start_station_name', 'end_station_name', 'start_lat', 'start_lng', 
                    'end_lat', 'end_lng', 'started_at', 'ended_at']
    
    existing_cols = [col for col in required_cols if col in trips_df.columns]
    trips_df = trips_df.dropna(subset=existing_cols)
    
    print("Converting timestamps to datetime with ROBUST parsing...")
    
    # =================================================================
    # FIXED TIMESTAMP PARSING - HANDLES MULTIPLE FORMATS
    # =================================================================
    
    def robust_timestamp_parser(timestamp_series):
        """Parse timestamps that may have different formats (with/without milliseconds)"""
        print(f"  Parsing {len(timestamp_series):,} timestamps...")
        
        # FIXED: COMBINED APPROACH - Parse with multiple formats and merge results
        
        # Create empty result series
        final_result = pd.Series([pd.NaT] * len(timestamp_series), index=timestamp_series.index)
        total_parsed = 0
        
        # Key formats for 2024 Citibike data (try in order of specificity)
        formats_to_try = [
            ('%Y-%m-%d %H:%M:%S.%f', 'with milliseconds'),     # 2024-12-06 17:50:49.428 (Jun-Dec)
            ('%Y-%m-%d %H:%M:%S', 'standard format'),          # 2024-01-15 15:18:07 (Jan-May)
            ('%Y-%m-%dT%H:%M:%S.%f', 'ISO with milliseconds'), # ISO format
            ('%Y-%m-%dT%H:%M:%S', 'ISO standard'),             # ISO format
        ]
        
        for fmt, description in formats_to_try:
            try:
                # Parse with this format
                parsed = pd.to_datetime(timestamp_series, format=fmt, errors='coerce')
                
                # Count new successful parses (not already parsed)
                newly_parsed_mask = parsed.notna() & final_result.isna()
                newly_parsed_count = newly_parsed_mask.sum()
                
                if newly_parsed_count > 0:
                    # Update final result with newly parsed timestamps
                    final_result.loc[newly_parsed_mask] = parsed.loc[newly_parsed_mask]
                    total_parsed += newly_parsed_count
                    print(f"    ✅ Format '{fmt}' ({description}): {newly_parsed_count:,} new successful parses")
                else:
                    print(f"    ➖ Format '{fmt}' ({description}): 0 new successful parses")
                    
            except Exception as e:
                print(f"    ❌ Format '{fmt}' failed: {e}")
        
        # Try auto-inference for any remaining timestamps
        if total_parsed < len(timestamp_series):
            remaining_mask = final_result.isna()
            remaining_count = remaining_mask.sum()
            
            if remaining_count > 0:
                print(f"    🔄 Trying auto-inference for remaining {remaining_count:,} timestamps...")
                try:
                    auto_parsed = pd.to_datetime(timestamp_series[remaining_mask], 
                                                errors='coerce', infer_datetime_format=True)
                    auto_success_mask = auto_parsed.notna()
                    auto_success_count = auto_success_mask.sum()
                    
                    if auto_success_count > 0:
                        # Update positions where auto-parsing worked
                        update_indices = remaining_mask[remaining_mask].index[auto_success_mask]
                        final_result.loc[update_indices] = auto_parsed[auto_success_mask]
                        total_parsed += auto_success_count
                        print(f"    ✅ Auto-inference: {auto_success_count:,} additional successful parses")
                    else:
                        print(f"    ➖ Auto-inference: 0 additional successful parses")
                        
                except Exception as e:
                    print(f"    ❌ Auto-inference failed: {e}")
        
        failed_count = len(timestamp_series) - total_parsed
        print(f"    📊 FINAL: {total_parsed:,} successful, {failed_count:,} failed")
        
        return final_result
    
    # Apply robust parsing to both timestamp columns
    print("Parsing started_at timestamps...")
    trips_df['started_at'] = robust_timestamp_parser(trips_df['started_at'])
    
    print("Parsing ended_at timestamps...")
    trips_df['ended_at'] = robust_timestamp_parser(trips_df['ended_at'])
    
    # Check results
    started_at_issues = trips_df['started_at'].isna().sum()
    ended_at_issues = trips_df['ended_at'].isna().sum()
    
    print(f"\nTimestamp parsing results:")
    print(f"  started_at successful: {len(trips_df) - started_at_issues:,}")
    print(f"  ended_at successful: {len(trips_df) - ended_at_issues:,}")
    print(f"  started_at failures: {started_at_issues:,}")
    print(f"  ended_at failures: {ended_at_issues:,}")
    
    # Remove rows with timestamp parsing failures
    original_count = len(trips_df)
    trips_df = trips_df.dropna(subset=['started_at', 'ended_at'])
    removed_count = original_count - len(trips_df)
    
    print(f"Removed {removed_count:,} rows with timestamp parsing issues")
    print(f"Remaining trips: {len(trips_df):,}")
    
    # Check final month distribution
    if len(trips_df) > 0:
        print("\nFinal month distribution after robust timestamp parsing:")
        monthly_counts = trips_df['started_at'].dt.month.value_counts().sort_index()
        month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        for month, count in monthly_counts.items():
            print(f"  {month_names[month]} ({month:02d}): {count:,} trips")
    
    # Continue with rest of processing
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                        (trips_df['duration_minutes'] < 24*60)]
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    print("Calculating station popularity...")
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    station_data = {}
    
    for station in top_stations:
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
        
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    stations_df = pd.DataFrame([
        {
            'name': data['name'],
            'lat': data['lat'],
            'lng': data['lng'],
            'popularity': data['popularity']
        }
        for idx, data in station_data.items()
    ])
    
    return station_mapping, station_data, top_stations, stations_df

def calculate_od_matrix(trips_df, station_mapping):
    """Calculate the origin-destination matrix from trips"""
    print("Calculating OD matrix...")
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    num_stations = len(station_mapping)
    od_matrix = pd.DataFrame(
        np.zeros((num_stations, num_stations)),
        index=range(num_stations),
        columns=range(num_stations)
    )
    
    for _, trip in mapped_trips.iterrows():
        origin_idx = station_mapping[trip['start_station_name']]
        dest_idx = station_mapping[trip['end_station_name']]
        od_matrix.iloc[origin_idx, dest_idx] += 1
    
    return od_matrix

###########################################
# Enhanced Visualization Functions with Draggable Legends
###########################################

def create_enhanced_station_map(stations_df, zone_classification, output_dir="results"):
    """Create enhanced interactive station map with BINARY zone classification legend"""
    print("Creating enhanced interactive station map...")
    
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Create map with default tile layer first
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,  # Start with no tiles
        control_scale=True
    )
    
    # Add multiple tile layers with proper control=True and attributions
    folium.TileLayer(
        'cartodbpositron', 
        name='CartoDB Positron (Light)', 
        control=True,
        show=True,  # Make this the default visible layer
        attr='© CartoDB, © OpenStreetMap'
    ).add_to(m)
    
    folium.TileLayer(
        'openstreetmap', 
        name='OpenStreetMap (Standard)', 
        control=True,
        show=False,
        attr='© OpenStreetMap contributors'
    ).add_to(m)
    
    folium.TileLayer(
        'cartodbdark_matter', 
        name='CartoDB Dark (Night Mode)', 
        control=True,
        show=False,
        attr='© CartoDB, © OpenStreetMap'
    ).add_to(m)
    
    folium.TileLayer(
        'stamenterrain',
        name='Terrain View',
        control=True,
        show=False,
        attr='© Stamen Design, © OpenStreetMap contributors'
    ).add_to(m)
    
    # Create draggable legend - ONLY residential and commercial
    legend_items = [
        {'type': 'circle', 'color': '#e74c3c', 'label': 'Residential Zones'},
        {'type': 'circle', 'color': '#3498db', 'label': 'Commercial Zones'}
    ]
    
    draggable_legend = create_draggable_html_legend(
        legend_items, 
        title="Jersey City Citibike Network (Binary Classification)", 
        position="top-right"
    )
    m.get_root().html.add_child(folium.Element(draggable_legend))
    
    # Create feature groups - ONLY two groups
    residential_group = folium.FeatureGroup(name="Residential Zones", show=True)
    commercial_group = folium.FeatureGroup(name="Commercial Zones", show=True)
    
    # Add stations with BINARY zone classification
    for _, station in stations_df.iterrows():
        zone_type = zone_classification.get(station['name'], 'residential')  # Default to residential
        
        if zone_type == 'residential':
            color = '#e74c3c'
            target_group = residential_group
        else:  # commercial
            color = '#3498db'
            target_group = commercial_group
        
        popup_html = f"""
        <strong>{station['name']}</strong><br>
        <strong>Zone Type:</strong> {zone_type.title()}<br>
        <strong>Total Trips:</strong> {station['popularity']:,}<br>
        <strong>Location:</strong> {station['lat']:.4f}, {station['lng']:.4f}
        """
        
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=8,
            color='black',
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=2,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(target_group)
    
    # Add neighborhood labels
    for neighborhood, (lat, lng) in NEIGHBORHOOD_LABELS.items():
        folium.Marker(
            location=[lat, lng],
            icon=folium.DivIcon(
                html=f'<div style="font-size: 12px; font-weight: bold; '
                     f'background-color: rgba(255,255,255,0.9); '
                     f'border: 1px solid black; padding: 2px 6px; '
                     f'border-radius: 3px; text-align: center;">{neighborhood}</div>',
                icon_size=(120, 25),
                icon_anchor=(60, 12)
            )
        ).add_to(m)
    
    # Add feature groups to map
    residential_group.add_to(m)
    commercial_group.add_to(m)
    
    # Add layer control - THIS IS ESSENTIAL for tile layer switching
    folium.LayerControl(position='topleft', collapsed=False).add_to(m)
    
    os.makedirs(output_dir, exist_ok=True)
    map_filename = os.path.join(output_dir, "binary_classification_station_map.html")
    m.save(map_filename)
    print(f"Binary classification station map saved to {map_filename}")
    
    return m

def create_enhanced_flow_map(od_matrix, stations_df, period_name, 
                           zone_classification=None, flow_scale=1.0, min_trips=5):
    """Create enhanced flow map with draggable legend showing BINARY zones and flow types"""
    print(f"Creating enhanced flow map: {period_name}")
    
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Create map with multiple tile layers and proper controls
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,  # Start with no default tiles
        control_scale=True
    )
    
    # Add multiple tile layers with proper control and attributions
    folium.TileLayer(
        'cartodbpositron', 
        name='Light Mode (CartoDB)', 
        control=True,
        show=True,  # Default visible layer
        attr='© CartoDB, © OpenStreetMap'
    ).add_to(m)
    
    folium.TileLayer(
        'openstreetmap', 
        name='Standard (OpenStreetMap)', 
        control=True,
        show=False,
        attr='© OpenStreetMap contributors'
    ).add_to(m)
    
    folium.TileLayer(
        'cartodbdark_matter', 
        name='Dark Mode (CartoDB)', 
        control=True,
        show=False,
        attr='© CartoDB, © OpenStreetMap'
    ).add_to(m)
    
    folium.TileLayer(
        'stamenterrain',
        name='Terrain View',
        control=True,
        show=False,
        attr='© Stamen Design, © OpenStreetMap contributors'
    ).add_to(m)
    
    # FIXED: Create BINARY classification draggable legend with IMPROVED COLORS
    legend_items = [
        {'type': 'circle', 'color': '#e74c3c', 'label': 'Residential Zones'},
        {'type': 'circle', 'color': '#3498db', 'label': 'Commercial Zones'},
        {'type': 'line', 'color': '#2ecc71', 'width': 6, 'label': 'High Flow (>80th percentile)'},
        {'type': 'line', 'color': '#f39c12', 'width': 4, 'label': 'Medium Flow (50-80th percentile)'},
        {'type': 'line', 'color': '#6595b5', 'width': 3, 'label': 'Low Flow (<50th percentile)'}  # CHANGED from light gray to purple
    ]
    
    draggable_legend = create_draggable_html_legend(
        legend_items, 
        title=f"Binary Flow Map: {period_name}", 
        position="top-right"
    )
    m.get_root().html.add_child(folium.Element(draggable_legend))
    
    # Create feature groups for different flow types
    high_flow_group = folium.FeatureGroup(name="High Flow Routes", show=True)
    medium_flow_group = folium.FeatureGroup(name="Medium Flow Routes", show=True)
    low_flow_group = folium.FeatureGroup(name="Low Flow Routes", show=True)
    
    # Add stations with BINARY zone classification colors only
    for _, station in stations_df.iterrows():
        if zone_classification and station['name'] in zone_classification:
            zone_type = zone_classification[station['name']]
            if zone_type == 'residential':
                color = '#e74c3c'
            else:  # commercial (only two options now)
                color = '#3498db'
        else:
            color = '#e74c3c'  # Default to residential
            zone_type = 'residential'
        
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=6,
            color='black',
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=2,
            popup=folium.Popup(
                f"<b>{station['name']}</b><br>"
                f"Zone Type: {zone_type.title()}<br>"
                f"Total Trips: {station['popularity']:,}",
                max_width=250
            )
        ).add_to(m)
    
    # Enhanced flow visualization with IMPROVED COLORS
    max_flow = od_matrix.max().max()
    flows = []
    
    for i in range(len(od_matrix)):
        for j in range(len(od_matrix)):
            if i != j and od_matrix.iloc[i, j] >= min_trips:
                flows.append(od_matrix.iloc[i, j])
    
    if flows:
        high_threshold = np.percentile(flows, 80)
        medium_threshold = np.percentile(flows, 50)
        
        for i in range(len(od_matrix)):
            for j in range(len(od_matrix)):
                if i != j and od_matrix.iloc[i, j] >= min_trips:
                    flow = od_matrix.iloc[i, j]
                    start_station = stations_df.iloc[i]
                    end_station = stations_df.iloc[j]
                    
                    if flow >= high_threshold:
                        color = '#2ecc71'  # Green for high flow
                        weight = max(4, min(8, flow / max_flow * 10))
                        opacity = 0.9
                        target_group = high_flow_group
                        flow_category = "High Flow"
                    elif flow >= medium_threshold:
                        color = '#f39c12'  # Orange for medium flow
                        weight = max(2, min(5, flow / max_flow * 8))
                        opacity = 0.7
                        target_group = medium_flow_group
                        flow_category = "Medium Flow"
                    else:
                        color = '#6595b5'  # Color for low flow
                        weight = max(2, min(4, flow / max_flow * 6))  # IMPROVED: Increased minimum weight
                        opacity = 0.4  # IMPROVED: Increased opacity for better visibility
                        target_group = low_flow_group
                        flow_category = "Low Flow"
                    
                    folium.PolyLine(
                        locations=[[start_station['lat'], start_station['lng']], 
                                 [end_station['lat'], end_station['lng']]],
                        color=color,
                        weight=weight,
                        opacity=opacity,
                        popup=folium.Popup(
                            f"<b>From:</b> {start_station['name']}<br>"
                            f"<b>To:</b> {end_station['name']}<br>"
                            f"<b>Trips:</b> {flow}<br>"
                            f"<b>Flow Category:</b> {flow_category}<br>"
                            f"<b>Flow Type:</b> {get_flow_type(start_station['name'], end_station['name'], zone_classification)}",
                            max_width=300
                        )
                    ).add_to(target_group)
    
    # Add feature groups to map
    high_flow_group.add_to(m)
    medium_flow_group.add_to(m)
    low_flow_group.add_to(m)
    
    # Add layer control - THIS IS ESSENTIAL for tile and feature layer switching
    folium.LayerControl(position='topleft', collapsed=False).add_to(m)
    
    return m

def create_enhanced_od_matrix_heatmap(od_matrix, stations_df, period_name, 
                                    zone_classification=None, output_dir="results"):
    """Create enhanced OD matrix heatmap with BINARY zone type indicators, better legend placement, and SVG export"""
    print(f"Creating enhanced OD matrix heatmap for {period_name}")
    
    # Create figure with much larger size for better font accommodation
    plt.figure(figsize=(22, 18))
    
    # Prepare station labels with BINARY zone indicators
    station_labels = []
    for _, station in stations_df.iterrows():
        name = station['name']
        if len(name) > 25:
            name = name[:25] + "..."
        
        if zone_classification and station['name'] in zone_classification:
            zone_type = zone_classification[station['name']]
            if zone_type == 'residential':
                prefix = "[R]"
            else:  # commercial (only two options)
                prefix = "[C]"
            station_labels.append(f"{prefix} {name}")
        else:
            station_labels.append(f"[R] {name}")  # Default to residential
    
    # Create heatmap with better spacing and larger colorbar
    ax = plt.subplot(111)
    sns.heatmap(
        od_matrix,
        cmap='YlOrRd',
        square=True,
        cbar_kws={'label': 'Number of Trips', 'shrink': 0.6, 'aspect': 20, 'pad': 0.02},
        linewidths=0.1,
        linecolor='white',
        xticklabels=station_labels,
        yticklabels=station_labels,
        ax=ax
    )
    
    # Enhanced title with MUCH larger fonts
    plt.title(f'Binary Classification OD Matrix: {period_name}\n'
              f'[R] = Residential Zone | [C] = Commercial Zone', 
              fontsize=22, pad=30, fontweight='bold')
    plt.xlabel('Destination Station', fontsize=18, labelpad=20, fontweight='bold')
    plt.ylabel('Origin Station', fontsize=18, labelpad=20, fontweight='bold')
    
    # MUCH larger station name fonts for better readability
    plt.xticks(rotation=45, ha='right', fontsize=14, fontweight='medium')
    plt.yticks(rotation=0, fontsize=14, fontweight='medium')
    
    # Make colorbar label larger
    cbar = ax.collections[0].colorbar
    cbar.set_label('Number of Trips', fontsize=16, fontweight='bold')
    cbar.ax.tick_params(labelsize=14)
    
    # Create BINARY legend with MUCH larger fonts and markers
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#e74c3c', 
               markersize=16, label='[R] Residential Zone', markeredgecolor='black', markeredgewidth=2),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#3498db', 
               markersize=16, label='[C] Commercial Zone', markeredgecolor='black', markeredgewidth=2)
    ]
    
    # Place legend with much larger fonts
    legend = make_matplotlib_legend_draggable(
        ax, 
        handles=legend_elements, 
        loc='upper left',
        fontsize=16,  # Much larger legend text
        title="Binary Zone Types",
        title_fontsize=18,  # Much larger legend title
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.95,
        facecolor='white',
        edgecolor='black'
    )
    
    # Add BINARY zone type summary with larger font
    if zone_classification:
        res_count = sum(1 for z in zone_classification.values() if z == 'residential')
        comm_count = sum(1 for z in zone_classification.values() if z == 'commercial')
        
        plt.figtext(0.02, 0.02, 
                   f"Binary Classification Summary: {res_count} Residential Zones, {comm_count} Commercial Zones (Total: {res_count + comm_count} stations)",
                   fontsize=14, style='italic', weight='bold',  # Larger summary text
                   bbox=dict(boxstyle="round,pad=0.6", facecolor="lightgray", alpha=0.8))
    
    # Adjust layout to prevent cutoff
    plt.tight_layout()
    
    # Save both PNG and SVG formats
    safe_name = period_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "to")
    
    # Save PNG version
    png_path = os.path.join(output_dir, f'binary_{safe_name}_od_matrix_heatmap.png')
    plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white', format='png')
    print(f"PNG heatmap saved to: {png_path}")
    
    # Save SVG version for Inkscape
    svg_path = os.path.join(output_dir, f'binary_{safe_name}_od_matrix_heatmap.svg')
    plt.savefig(svg_path, bbox_inches='tight', facecolor='white', format='svg')
    print(f"SVG heatmap saved to: {svg_path}")
    
    plt.close()
    
    return png_path, svg_path


###########################################
# Additional Analysis Functions with Draggable Legends
###########################################

def create_flow_direction_analysis(trips_df, station_mapping, zone_classification, output_dir):
    """Create comprehensive flow direction analysis with draggable legends and dual format export"""
    print("Creating flow direction analysis...")
    
    fig, axes = plt.subplots(2, 2, figsize=(22, 18))  # Increased size
    fig.suptitle('Flow Direction Analysis: Residential ↔ Commercial Patterns', 
                 fontsize=20, fontweight='bold', y=0.95)
    
    periods = list(TIME_PERIODS.keys())
    
    for idx, (period_name, (start_hour, end_hour)) in enumerate(TIME_PERIODS.items()):
        ax = axes[idx // 2, idx % 2]
        
        if period_name == 'All Day':
            period_trips = trips_df
        else:
            period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                  (trips_df['start_hour'] < end_hour)]
        
        flow_counts = {'Res→Comm': 0, 'Comm→Res': 0, 'Res→Res': 0, 'Comm→Comm': 0}
        
        for _, trip in period_trips.iterrows():
            origin_station = trip['start_station_name']
            dest_station = trip['end_station_name']
            
            if origin_station in station_mapping and dest_station in station_mapping:
                origin_type = zone_classification.get(origin_station, 'unknown')
                dest_type = zone_classification.get(dest_station, 'unknown')
                
                if origin_type == 'residential' and dest_type == 'commercial':
                    flow_counts['Res→Comm'] += 1
                elif origin_type == 'commercial' and dest_type == 'residential':
                    flow_counts['Comm→Res'] += 1
                elif origin_type == 'residential' and dest_type == 'residential':
                    flow_counts['Res→Res'] += 1
                elif origin_type == 'commercial' and dest_type == 'commercial':
                    flow_counts['Comm→Comm'] += 1
        
        flow_types = list(flow_counts.keys())
        values = list(flow_counts.values())
        colors = ['#e74c3c', '#3498db', '#e67e22', '#9b59b6']
        
        bars = ax.bar(flow_types, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        
        # Add percentage labels
        total = sum(values)
        for bar, value in zip(bars, values):
            height = bar.get_height()
            percentage = (value / total * 100) if total > 0 else 0
            ax.text(bar.get_x() + bar.get_width()/2., height + total*0.01,
                   f'{percentage:.1f}%\n({value:,})',
                   ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        # Highlight dominant flow
        if values:
            max_idx = values.index(max(values))
            bars[max_idx].set_edgecolor('gold')
            bars[max_idx].set_linewidth(4)
        
        ax.set_title(period_name, fontsize=16, fontweight='bold', pad=15)
        ax.set_ylabel('Number of Trips', fontsize=13)
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.tick_params(axis='y', labelsize=11)
        ax.grid(axis='y', alpha=0.3)
        
        # Better legend placement for each subplot
        legend_elements = [
            mpatches.Rectangle((0, 0), 1, 1, facecolor='#e74c3c', alpha=0.8, 
                             edgecolor='black', label='Residential → Commercial'),
            mpatches.Rectangle((0, 0), 1, 1, facecolor='#3498db', alpha=0.8, 
                             edgecolor='black', label='Commercial → Residential'),
            mpatches.Rectangle((0, 0), 1, 1, facecolor='#e67e22', alpha=0.8, 
                             edgecolor='black', label='Residential → Residential'),
            mpatches.Rectangle((0, 0), 1, 1, facecolor='#9b59b6', alpha=0.8, 
                             edgecolor='black', label='Commercial → Commercial')
        ]
        
        make_matplotlib_legend_draggable(
            ax, 
            handles=legend_elements, 
            loc='upper right', 
            fontsize=9,
            frameon=True,
            fancybox=True,
            shadow=True,
            framealpha=0.95
        )
        
        if values:
            ax.set_ylim(0, max(values) * 1.25)
    
    plt.tight_layout()
    
    # Save both formats
    base_name = 'flow_direction_analysis'
    png_path = os.path.join(output_dir, f'{base_name}.png')
    svg_path = os.path.join(output_dir, f'{base_name}.svg')
    
    plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white', format='png')
    plt.savefig(svg_path, bbox_inches='tight', facecolor='white', format='svg')
    plt.close()
    
    print(f"Flow direction analysis saved to:")
    print(f"  PNG: {png_path}")
    print(f"  SVG: {svg_path}")
    return png_path, svg_path

def create_network_topology_overview(stations_df, zone_classification, output_dir):
    """Create network topology overview with BINARY classification, better legend, and dual format export"""
    print("Creating network topology overview...")
    
    fig, ax = plt.subplots(figsize=(16, 12))  # Increased size
    
    # Plot stations by BINARY zone type only
    residential_stations = []
    commercial_stations = []
    
    for _, station in stations_df.iterrows():
        zone_type = zone_classification.get(station['name'], 'residential')
        
        if zone_type == 'residential':
            residential_stations.append(station)
        else:  # commercial
            commercial_stations.append(station)
    
    # Plot residential stations
    if residential_stations:
        res_df = pd.DataFrame(residential_stations)
        ax.scatter(res_df['lng'], res_df['lat'], 
                  c='#e74c3c', marker='o', s=120, 
                  alpha=0.8, edgecolors='black', linewidth=1.5,
                  label='Residential Zones')
    
    # Plot commercial stations  
    if commercial_stations:
        comm_df = pd.DataFrame(commercial_stations)
        ax.scatter(comm_df['lng'], comm_df['lat'], 
                  c='#3498db', marker='s', s=140, 
                  alpha=0.8, edgecolors='black', linewidth=1.5,
                  label='Commercial Zones')
    
    # Add neighborhood labels with better styling
    for name, (lat, lng) in NEIGHBORHOOD_LABELS.items():
        ax.annotate(name, (lng, lat), 
                   xytext=(8, 8), textcoords='offset points',
                   fontsize=13, fontweight='bold',
                   bbox=dict(boxstyle="round,pad=0.4", facecolor="white", 
                           alpha=0.9, edgecolor='black', linewidth=1))
    
    # Create BINARY legend with better placement
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', 
               markersize=14, label='Residential Zone', markeredgecolor='black'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#3498db', 
               markersize=14, label='Commercial Zone', markeredgecolor='black')
    ]
    
    # Place legend in bottom-right corner
    make_matplotlib_legend_draggable(
        ax, 
        handles=legend_elements, 
        loc='lower right',
        fontsize=14,
        title="Binary Zone Classification",
        title_fontsize=16,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.95,
        facecolor='white',
        edgecolor='black'
    )
    
    ax.set_title('Jersey City Citibike Network: Binary Zone Classification\n(Residential vs Commercial)', 
                fontsize=18, fontweight='bold', pad=25)
    ax.set_xlabel('Longitude', fontsize=14, labelpad=10)
    ax.set_ylabel('Latitude', fontsize=14, labelpad=10)
    ax.grid(True, alpha=0.4, linestyle='--')
    
    # Keep coordinate labels but make them smaller
    ax.tick_params(axis='both', labelsize=10)
    
    plt.tight_layout()
    
    # Save both formats
    base_name = 'binary_network_topology_overview'
    png_path = os.path.join(output_dir, f'{base_name}.png')
    svg_path = os.path.join(output_dir, f'{base_name}.svg')
    
    plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white', format='png')
    plt.savefig(svg_path, bbox_inches='tight', facecolor='white', format='svg')
    plt.close()
    
    print(f"Binary network topology overview saved to:")
    print(f"  PNG: {png_path}")
    print(f"  SVG: {svg_path}")
    return png_path, svg_path

def analyze_seasonal_patterns(trips_df, output_dir):
    """Analyze seasonal patterns with better legend placement and dual format export"""
    print("Analyzing seasonal patterns...")
    
    seasonal_dir = os.path.join(output_dir, 'seasonal_analysis')
    os.makedirs(seasonal_dir, exist_ok=True)
    
    seasons = {
        'winter': [1, 2, 12],
        'spring': [3, 4, 5], 
        'summer': [6, 7, 8],
        'fall': [9, 10, 11]
    }
    
    if 'month' not in trips_df.columns:
        print("No month data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['month'] = pd.to_datetime(trips_df['started_at']).dt.month
        else:
            print("Cannot analyze seasonal patterns: no month or datetime data available")
            return seasonal_dir
    
    available_months = trips_df['month'].unique()
    print(f"Available months: {sorted(available_months)}")
    
    monthly_trips = trips_df.groupby('month').size()
    
    seasonal_trips = {}
    for season_name, season_months in seasons.items():
        valid_months = [m for m in season_months if m in available_months]
        if valid_months:
            seasonal_trips[season_name] = monthly_trips[valid_months].sum()
    
    fig, ax = plt.subplots(figsize=(14, 10))  # Increased size
    
    season_colors = {
        'winter': '#A1D6E2',
        'spring': '#81C784',
        'summer': '#FFB74D',
        'fall': '#E57373'
    }
    
    bars = ax.bar(
        range(len(seasonal_trips)),
        list(seasonal_trips.values()),
        width=0.7,
        color=[season_colors.get(s, '#BBBBBB') for s in seasonal_trips.keys()],
        edgecolor='black',
        linewidth=1.5,
        alpha=0.9
    )
    
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=14,
            fontweight='bold'
        )
    
    ax.set_title('Citibike Trips by Season', fontsize=18, pad=25, fontweight='bold')
    ax.set_xlabel('Season', fontsize=15, labelpad=15)
    ax.set_ylabel('Number of Trips', fontsize=15, labelpad=15)
    ax.set_xticks(range(len(seasonal_trips)))
    ax.set_xticklabels([s.capitalize() for s in seasonal_trips.keys()], fontsize=14)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='y', labelsize=12)
    
    # Better legend placement
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, facecolor=season_colors['winter'], 
                         edgecolor='black', label='Winter'),
        mpatches.Rectangle((0, 0), 1, 1, facecolor=season_colors['spring'], 
                         edgecolor='black', label='Spring'),
        mpatches.Rectangle((0, 0), 1, 1, facecolor=season_colors['summer'], 
                         edgecolor='black', label='Summer'),
        mpatches.Rectangle((0, 0), 1, 1, facecolor=season_colors['fall'], 
                         edgecolor='black', label='Fall')
    ]
    
    make_matplotlib_legend_draggable(
        ax, 
        handles=legend_elements, 
        loc='upper left',
        fontsize=13,
        title="Seasons",
        title_fontsize=15,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.95
    )
    
    plt.tight_layout()
    
    # Save both formats
    base_name = 'seasonal_trip_counts'
    png_path = os.path.join(seasonal_dir, f'{base_name}.png')
    svg_path = os.path.join(seasonal_dir, f'{base_name}.svg')
    
    plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white', format='png')
    plt.savefig(svg_path, bbox_inches='tight', facecolor='white', format='svg')
    plt.close()
    
    print(f"Seasonal analysis saved to:")
    print(f"  PNG: {png_path}")
    print(f"  SVG: {svg_path}")
    return seasonal_dir

def analyze_day_type_patterns(trips_df, output_dir):
    """Analyze weekday vs weekend patterns with better legend and dual format export"""
    print("Analyzing weekday vs weekend patterns...")
    
    day_type_dir = os.path.join(output_dir, 'day_type_analysis')
    os.makedirs(day_type_dir, exist_ok=True)
    
    if 'is_weekend' not in trips_df.columns and 'start_day' not in trips_df.columns:
        print("No day type data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['start_day'] = pd.to_datetime(trips_df['started_at']).dt.dayofweek
            trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
        else:
            print("Cannot analyze day type patterns: no day or datetime data available")
            return day_type_dir
    
    if 'is_weekend' not in trips_df.columns and 'start_day' in trips_df.columns:
        trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    day_type_colors = {
        'weekday': '#3498db',
        'weekend': '#e74c3c'
    }
    
    day_type_counts = trips_df.groupby('is_weekend').size()
    day_type_labels = ['Weekday', 'Weekend']
    
    fig, ax = plt.subplots(figsize=(12, 10))  # Increased size
    
    bars = ax.bar(
        day_type_labels,
        [day_type_counts.get(0, 0), day_type_counts.get(1, 0)],
        width=0.6,
        color=[day_type_colors['weekday'], day_type_colors['weekend']],
        edgecolor='black',
        linewidth=1.5,
        alpha=0.9
    )
    
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=16,
            fontweight='bold'
        )
    
    ax.set_title('Citibike Trips: Weekday vs Weekend', fontsize=18, pad=25, fontweight='bold')
    ax.set_ylabel('Number of Trips', fontsize=15, labelpad=15)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='x', labelsize=15)
    ax.tick_params(axis='y', labelsize=13)
    
    # Better legend placement
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, facecolor=day_type_colors['weekday'], 
                         edgecolor='black', label='Weekday'),
        mpatches.Rectangle((0, 0), 1, 1, facecolor=day_type_colors['weekend'], 
                         edgecolor='black', label='Weekend')
    ]
    
    make_matplotlib_legend_draggable(
        ax, 
        handles=legend_elements, 
        loc='upper left',
        fontsize=14,
        title="Day Types",
        title_fontsize=16,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.95
    )
    
    plt.tight_layout()
    
    # Save both formats
    base_name = 'day_type_trip_counts'
    png_path = os.path.join(day_type_dir, f'{base_name}.png')
    svg_path = os.path.join(day_type_dir, f'{base_name}.svg')
    
    plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white', format='png')
    plt.savefig(svg_path, bbox_inches='tight', facecolor='white', format='svg')
    plt.close()
    
    print(f"Day type analysis saved to:")
    print(f"  PNG: {png_path}")
    print(f"  SVG: {svg_path}")
    return day_type_dir

def create_summary_report(trips_df, stations_df, station_mapping, zone_classification, output_dir):
    """Create summary report for BINARY classification"""
    print("Creating binary classification summary report...")
    
    report_path = os.path.join(output_dir, 'binary_classification_summary.txt')
    
    # Calculate statistics
    total_trips = len(trips_df)
    total_stations = len(stations_df)
    date_range = f"{trips_df['started_at'].min().strftime('%Y-%m-%d')} to {trips_df['started_at'].max().strftime('%Y-%m-%d')}"
    
    # Zone type statistics - ONLY residential and commercial
    zone_counts = {}
    for zone_type in ['residential', 'commercial']:
        zone_counts[zone_type] = sum(1 for z in zone_classification.values() if z == zone_type)
    
    # Calculate percentage split
    total_classified = sum(zone_counts.values())
    residential_pct = (zone_counts['residential'] / total_classified * 100) if total_classified > 0 else 0
    commercial_pct = (zone_counts['commercial'] / total_classified * 100) if total_classified > 0 else 0
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("="*90 + "\n")
        f.write("           BINARY CITIBIKE ZONE CLASSIFICATION: RESIDENTIAL vs COMMERCIAL\n")
        f.write("="*90 + "\n\n")
        
        f.write("DATASET OVERVIEW\n")
        f.write("-"*50 + "\n")
        f.write(f"Total Trips Analyzed: {total_trips:,}\n")
        f.write(f"Number of Stations: {total_stations}\n")
        f.write(f"Date Range: {date_range}\n\n")
        
        f.write("BINARY ZONE CLASSIFICATION SUMMARY\n")
        f.write("-"*50 + "\n")
        f.write(f"Residential Zones: {zone_counts['residential']} stations ({residential_pct:.1f}%)\n")
        f.write(f"Commercial Zones: {zone_counts['commercial']} stations ({commercial_pct:.1f}%)\n")
        f.write(f"Total Classified: {total_classified} stations\n\n")
        
        # Flow analysis
        flow_analysis = analyze_flow_types(trips_df, zone_classification)
        f.write("RESIDENTIAL ↔ COMMERCIAL FLOW ANALYSIS\n")
        f.write("-"*50 + "\n")
        total_flows = sum(flow_analysis.values())
        for flow_type, count in flow_analysis.items():
            if flow_type != 'Other':  # Skip 'Other' category since we have binary classification
                percentage = (count / total_flows * 100) if total_flows > 0 else 0
                f.write(f"{flow_type}: {count:,} trips ({percentage:.1f}%)\n")
        
    print(f"Binary classification summary saved to {report_path}")

def analyze_flow_types(trips_df, zone_classification):
    """Analyze flow types for BINARY classification"""
    flow_types = {
        'Residential → Commercial': 0,
        'Commercial → Residential': 0,
        'Residential → Residential': 0,
        'Commercial → Commercial': 0
    }
    
    for _, trip in trips_df.iterrows():
        origin_type = zone_classification.get(trip['start_station_name'], 'residential')
        dest_type = zone_classification.get(trip['end_station_name'], 'residential')
        
        if origin_type == 'residential' and dest_type == 'commercial':
            flow_types['Residential → Commercial'] += 1
        elif origin_type == 'commercial' and dest_type == 'residential':
            flow_types['Commercial → Residential'] += 1
        elif origin_type == 'residential' and dest_type == 'residential':
            flow_types['Residential → Residential'] += 1
        else:  # commercial to commercial
            flow_types['Commercial → Commercial'] += 1
    
    return flow_types

###########################################
# Main Processing Function
###########################################

def process_citibike_data_enhanced(data_files_pattern, config=None, mode='all'):
    """Enhanced version of the main processing function with draggable legends"""
    if config is None:
        config = RUNTIME_CONFIG
    
    os.makedirs(config['OUTPUT_DIR'], exist_ok=True)
    
    print("Starting Enhanced Citibike OD Analysis with Draggable Legends")
    print(f"Mode: {mode}")
    print(f"Max stations: {config['MAX_STATIONS']}")
    print(f"Sample rate: {config['SAMPLE_RATE']}")
    
    try:
        # Load data with memory management
        trips_df = load_citibike_data(
            config['DATA_DIR'], 
            config['DATA_FILES_PATTERN'],
            sample_rate=config['SAMPLE_RATE'],
            chunk_size=config['CHUNK_SIZE']
        )
        
        # Get top stations and create station mapping
        station_mapping, station_data, top_station_names, stations_df = get_top_stations(
            trips_df, config['MAX_STATIONS']
        )
        
        # Add zone classification
        zone_classification = classify_stations_by_zone_type(stations_df)
        print(f"Zone classification completed:")
        zone_counts = {}
        for zone_type in ['residential', 'commercial']:
            count = sum(1 for z in zone_classification.values() if z == zone_type)
            zone_counts[zone_type] = count
            print(f"  {zone_type.title()}: {count} stations")
        
        print(f"Working with {len(station_mapping)} stations")
        
        # Run visualizations if requested
        if mode in ['visualize', 'all']:
            print("\nGenerating enhanced visualizations with draggable legends...")
            
            # Create enhanced interactive station map
            enhanced_station_map = create_enhanced_station_map(
                stations_df,
                zone_classification,
                output_dir=config['OUTPUT_DIR']
            )
            
            # Create enhanced flow maps for different time periods
            time_periods_dir = os.path.join(config['OUTPUT_DIR'], 'time_periods')
            os.makedirs(time_periods_dir, exist_ok=True)
            
            for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
                print(f"Processing enhanced {period_name} with hours {start_hour}-{end_hour}")
                
                period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                      (trips_df['start_hour'] < end_hour)]
                
                print(f"Processing enhanced {period_name} with {len(period_trips)} trips")
                
                if len(period_trips) == 0:
                    print(f"No trips found for {period_name}, skipping...")
                    continue
                    
                od_matrix = calculate_od_matrix(period_trips, station_mapping)
                
                # Create enhanced flow map
                flow_map = create_enhanced_flow_map(
                    od_matrix, 
                    stations_df,
                    period_name,
                    zone_classification=zone_classification,
                    flow_scale=3.0,
                    min_trips=5
                )
                
                safe_period_name = period_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "to")
                map_filename = f"enhanced_flow_map_{safe_period_name}.html"
                flow_map.save(os.path.join(time_periods_dir, map_filename))
                print(f"Enhanced flow map for {period_name} saved to {os.path.join(time_periods_dir, map_filename)}")
                
                # Clear memory
                del period_trips, od_matrix, flow_map
                gc.collect()
            
            # Create enhanced OD matrix heatmaps for each time period
            print("\nCreating enhanced OD matrix heatmaps...")
            heatmap_dir = os.path.join(config['OUTPUT_DIR'], 'enhanced_od_matrix_heatmaps')
            os.makedirs(heatmap_dir, exist_ok=True)
            
            for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
                period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                      (trips_df['start_hour'] < end_hour)]
                
                if len(period_trips) > 0:
                    od_matrix = calculate_od_matrix(period_trips, station_mapping)
                    create_enhanced_od_matrix_heatmap(
                        od_matrix, stations_df, period_name, 
                        zone_classification, heatmap_dir
                    )
                    
                    # Clear memory
                    del period_trips, od_matrix
                    gc.collect()
            
            # Create additional analysis visualizations
            print("\nCreating additional analysis visualizations...")
            
            # Flow direction analysis
            create_flow_direction_analysis(trips_df, station_mapping, zone_classification, config['OUTPUT_DIR'])
            
            # Network topology overview
            create_network_topology_overview(stations_df, zone_classification, config['OUTPUT_DIR'])
            
            # Seasonal and temporal analysis
            if config.get('ANALYZE_SEASONAL_PATTERNS', True):
                analyze_seasonal_patterns(trips_df, config['OUTPUT_DIR'])
            
            analyze_day_type_patterns(trips_df, config['OUTPUT_DIR'])
            
            print("\nAll enhanced visualizations with draggable legends completed!")
        
        # Generate enhanced summary report
        print("\nGenerating enhanced summary statistics...")
        create_summary_report(trips_df, stations_df, station_mapping, zone_classification, config['OUTPUT_DIR'])
        
        print("\nEnhanced analysis with draggable legends completed successfully!")
        return True
        
    except Exception as e:
        print(f"Error during enhanced analysis: {e}")
        import traceback
        traceback.print_exc()
        return False

def main():
    """Main entry point for Enhanced Citibike OD Matrix Analysis with Draggable Legends"""
    print("Starting enhanced analysis with draggable legends parameters:")
    print({
        'data_dir': RUNTIME_CONFIG['DATA_DIR'],
        'output_dir': RUNTIME_CONFIG['OUTPUT_DIR'],
        'max_stations': RUNTIME_CONFIG['MAX_STATIONS'],
        'sample_rate': RUNTIME_CONFIG['SAMPLE_RATE']
    })
    
    full_pattern = os.path.join(RUNTIME_CONFIG['DATA_DIR'], RUNTIME_CONFIG['DATA_FILES_PATTERN'])
    
    print(f"Data files pattern: {full_pattern}")
    print(f"Output directory: {RUNTIME_CONFIG['OUTPUT_DIR']}")
    print(f"Mode: {RUNTIME_CONFIG['MODE']}")
    
    try:
        success = process_citibike_data_enhanced(full_pattern, RUNTIME_CONFIG, RUNTIME_CONFIG['MODE'])
        if success:
            print("\n[SUCCESS] Enhanced analysis with draggable legends completed successfully!")
            print(f"[FOLDER] Check results in: {RUNTIME_CONFIG['OUTPUT_DIR']}")
            print("[INFO] Open the enhanced HTML files in a web browser to view interactive maps")
            print("[INFO] Enhanced features with draggable legends include:")
            print("  - Zone classification with draggable HTML legends (Red=Residential, Blue=Commercial")
            print("  - Enhanced flow visualizations with draggable flow type legends")
            print("  - Flow direction analysis charts with moveable legends")
            print("  - Network topology overview with draggable marker legends")
            print("  - Enhanced OD matrix heatmaps with draggable matplotlib legends")
            print("  - Seasonal and temporal analysis with fully draggable legends")
            print("[DRAG] All legends can be moved by clicking and dragging!")
        else:
            print("\n[ERROR] Enhanced analysis failed. Check error messages above.")
        return 0 if success else 1
    except FileNotFoundError as e:
        print(f"[ERROR] File not found: {e}")
        return 1
    except MemoryError as e:
        print(f"[ERROR] Memory error: {e}")
        print("[TIP] Try reducing SAMPLE_RATE or MAX_STATIONS in RUNTIME_CONFIG")
        return 1
    except Exception as e:
        print(f"[ERROR] Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        return 1

if __name__ == "__main__":
    main()

Starting enhanced analysis with draggable legends parameters:
{'data_dir': 'data/citibike', 'output_dir': 'results', 'max_stations': 20, 'sample_rate': 1.0}
Data files pattern: data/citibike\JC2024*citibiketripdata.csv
Output directory: results
Mode: all
Starting Enhanced Citibike OD Analysis with Draggable Legends
Mode: all
Max stations: 20
Sample rate: 1.0
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
  Processed 10 chunks...
Loa